In [ ]:
# ── 0.1  Mount Google Drive and clone/pull the repository ─────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir('/content')
!git clone https://github.com/VAL-Jerono/KHS_housing_dissertation.git 2>/dev/null || \
    (cd KHS_housing_dissertation && git pull)
os.chdir('KHS_housing_dissertation')
sys.path.insert(0, 'src')
print("✓ Drive mounted. Repository ready.")


In [ ]:
# ── 0.2  Install all required dependencies (first run only — ~90 seconds) ──────
!pip install -q polars pyarrow scikit-learn matplotlib seaborn scipy \
    xgboost lightgbm shap pytorch-tabnet geopandas statsmodels \
    mapclassify contextily imbalanced-learn joblib upsetplot
print("✓ All packages installed.")


In [ ]:
# ── 0.3  Core imports ─────────────────────────────────────────────────────────
import json, warnings, pickle
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import seaborn as sns
from pathlib import Path
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ── Sklearn ────────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (StratifiedKFold, StratifiedGroupKFold,
                                      KFold, cross_val_predict,
                                      RandomizedSearchCV, train_test_split)
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                              classification_report, confusion_matrix,
                              mean_squared_error, r2_score, mean_absolute_error,
                              precision_recall_curve, roc_curve)
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.calibration import calibration_curve
from sklearn.impute import SimpleImputer

# ── Boosting + Deep Learning ───────────────────────────────────────────────────
import xgboost as xgb
import lightgbm as lgb
import shap
import torch
import torch.nn as nn
from pytorch_tabnet.tab_model import TabNetRegressor, TabNetClassifier

import joblib
warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
print(f"✓ All imports loaded.")
print(f"  XGBoost {xgb.__version__} | LightGBM {lgb.__version__} | SHAP {shap.__version__}")
print(f"  PyTorch {torch.__version__}")


In [ ]:
# ── 0.4  Paths, constants, colour palette, plot style, county map ─────────────

DRIVE  = Path('/content/drive/MyDrive/KHS_Dissertation')
PQ     = DRIVE / 'data' / 'parquet'
RAW    = DRIVE / 'data' / 'raw'
OUT    = DRIVE / 'outputs'
FIGS   = OUT / 'figures'
TABS   = OUT / 'tables'
MODS   = OUT / 'models'
SHPS   = DRIVE / 'data' / 'shapefiles'
for p in [FIGS, TABS, MODS, SHPS]: p.mkdir(parents=True, exist_ok=True)

# ── Global constants ───────────────────────────────────────────────────────────
N_FOLDS        = 5       # Cross-validation folds
SEED           = 42      # Random seed — propagated everywhere
HFVS_THRESHOLD = 0.60    # Score above which a household is classified "high vulnerability"

# ── Colour palette (defined once, used in every chart) ─────────────────────────
TEAL   = '#00695C'
RED    = '#B71C1C'
AMBER  = '#E65100'
BLUE   = '#1565C0'
PURPLE = '#6A1B9A'
GRAY   = '#546E7A'
DARK   = '#2C2C2A'

# ── Publication plot style ─────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'        : 140,
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : '#F8F8F6',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.titlesize'    : 13,
    'axes.titleweight'  : '600',
    'axes.labelsize'    : 11,
    'xtick.labelsize'   : 9,
    'ytick.labelsize'   : 9,
    'font.family'       : 'sans-serif',
    'legend.framealpha' : 0.9,
    'legend.fontsize'   : 9,
})

# ── All 47 Kenya counties (survey integer code → county name string) ─────────
COUNTY_MAP = {
     1:'Mombasa',        2:'Kwale',          3:'Kilifi',         4:'Tana River',
     5:'Lamu',           6:'Taita-Taveta',   7:'Garissa',        8:'Wajir',
     9:'Mandera',       10:'Marsabit',      11:'Isiolo',        12:'Meru',
    13:'Tharaka-Nithi', 14:'Embu',          15:'Kitui',         16:'Machakos',
    17:'Makueni',       18:'Nyandarua',     19:'Nyeri',         20:'Kirinyaga',
    21:"Murang'a",      22:'Kiambu',        23:'Turkana',       24:'West Pokot',
    25:'Samburu',       26:'Trans Nzoia',   27:'Uasin Gishu',   28:'Elgeyo-Marakwet',
    29:'Nandi',         30:'Baringo',       31:'Laikipia',      32:'Nakuru',
    33:'Narok',         34:'Kajiado',       35:'Kericho',       36:'Bomet',
    37:'Kakamega',      38:'Vihiga',        39:'Bungoma',       40:'Busia',
    41:'Siaya',         42:'Kisumu',        43:'Homa Bay',      44:'Migori',
    45:'Kisii',         46:'Nyamira',       47:'Nairobi',
}

print(f"✓ Environment configured.")
print(f"  FIGS: {FIGS}")
print(f"  TABS: {TABS}")
print(f"  Counties defined: {len(COUNTY_MAP)}")

In [ ]:
# ── 1.4  Visualise the HFVS Framework ─────────────────────────────────────────
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')

BOX_W, BOX_H = 1.9, 1.2

# Central HFVS box
ax.add_patch(mpatches.Rectangle((4.05, 2.0), BOX_W, 1.4,
             fc='#1a237e', ec='white', lw=2, zorder=3))
ax.text(5.0, 2.7, 'HFVS\nComposite', ha='center', va='center',
        color='white', fontsize=11, fontweight='bold', zorder=4)

# Five dimension boxes
dims = [
    (0.3,  4.8, 'D1 Financial\nStress',    RED   ),   # top-left
    (0.3,  2.8, 'D2 Tenure\nInsecurity',   AMBER ),   # bottom-left
    (4.05, 5.8, 'D3 Physical\nHazard',     PURPLE),   # top-centre
    (7.8,  4.8, 'D4 Dwelling\nQuality',    BLUE  ),   # top-right
    (7.8,  2.8, 'D5 Utility\nDeprivation', TEAL  ),   # bottom-right
]

for bx, by, label, color in dims:
    ax.add_patch(mpatches.FancyBboxPatch((bx, by - BOX_H), BOX_W, BOX_H,
                 boxstyle='round,pad=0.08', fc=color, ec='white', lw=1.5, alpha=0.88))
    ax.text(bx + BOX_W/2, by - BOX_H/2, label,
            ha='center', va='center', color='white', fontsize=9, fontweight='bold')

    # Arrow routing —
    if bx < 4:        # left → right edge to left face of HFVS
        x_start, y_start = bx + BOX_W,   by - BOX_H/2
        x_end,   y_end   = 4.05,          2.7
    elif bx > 6:      # right → left edge to right face of HFVS
        x_start, y_start = bx,            by - BOX_H/2
        x_end,   y_end   = 4.05 + BOX_W,  2.7
    else:             # D3 top-centre → bottom edge down to top face of HFVS
        x_start, y_start = bx + BOX_W/2,  by - BOX_H
        x_end,   y_end   = 5.0,            2.0 + 1.4

    ax.annotate('', xy=(x_end, y_end), xytext=(x_start, y_start),
                arrowprops=dict(arrowstyle='->', color=GRAY, lw=1.5,
                                connectionstyle='arc3,rad=0.0'))

# Title & formula
ax.text(5.0, 5.75, 'HFVS = (D1 + D2 + D3 + D4 + D5) / 5     [equal weights]',
        ha='center', va='center', fontsize=10, fontweight='600', color=DARK,
        bbox=dict(boxstyle='round', fc='#F0F4F8', ec=GRAY, pad=0.4))
ax.set_title('Housing Financial Vulnerability Score — Five-Dimension Framework',
             fontsize=12, fontweight='700', pad=12)

plt.tight_layout()
plt.savefig(FIGS / 'phase1_hfvs_framework.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 2.1  Load codebook labels ─────────────────────────────────────────────────
# The KHS uses Stata-style integer codes for categorical variables.
# The JSON label files map code → label string for audit and interpretation.

with open(PQ / 'household_variable_labels.json')  as f: HH_VAR  = json.load(f)
with open(PQ / 'household_value_labels.json')     as f: HH_VAL  = json.load(f)
with open(PQ / 'dwelling_variable_labels.json')   as f: DW_VAR  = json.load(f)
with open(PQ / 'dwelling_value_labels.json')      as f: DW_VAL  = json.load(f)
with open(PQ / 'individual_variable_labels.json') as f: IND_VAR = json.load(f)

def decode(col, series, val_dict=HH_VAL):
    mapping = val_dict.get(col.upper(), val_dict.get(col, {}))
    return series.map(lambda x: mapping.get(str(int(x)), str(x)) if pd.notna(x) else np.nan)

print("✓ Codebook labels loaded.")
print(f"  Household variable labels : {len(HH_VAR):,}")
print(f"  Household value labels    : {len(HH_VAL):,}")
print(f"  Dwelling variable labels  : {len(DW_VAR):,}")

In [ ]:
# ── 2.2  File inventory — establish the full data universe ────────────────────
FILES = {
    'household' : 'Household_Information_Data.parquet',
    'individual': 'Individual_Data.parquet',
    'dwelling'  : 'Dwelling_Units_Data.parquet',
    'county'    : 'County_Physical_Planning_Data.parquet',
    'mortgage'  : 'Housing_Mortgage_Data.parquet',
    'loan'      : 'Housing_Loans_Data.parquet',
}

DESCRIPTIONS = {
    'household' : 'SPINE — 392 cols: finances, tenure, utilities, infrastructure',
    'individual': 'One row per person — demographics, education, migration, employment',
    'dwelling'  : 'Physical structure details — materials, rooms, floor area',
    'county'    : '47 rows — county-level planning data, infrastructure indicators',
    'mortgage'  : 'Mortgage records for borrowing households',
    'loan'      : 'Housing loan records',
}

print(f"  {'File':<13} {'Rows':>8} {'Cols':>6}  Description")
print("  " + "─" * 72)
dfs = {}
for key, fname in FILES.items():
    path = PQ / fname
    if not path.exists():
        print(f"  {'⚠ '+key:<13} {'—':>8} {'—':>6}  Not found")
        continue
    df = pl.read_parquet(path).to_pandas()
    dfs[key] = df
    print(f"  {key:<13} {df.shape[0]:>8,} {df.shape[1]:>6}  {DESCRIPTIONS.get(key,'')}")

hh       = dfs.get('household')
ind      = dfs.get('individual')
dw       = dfs.get('dwelling')
cnt      = dfs.get('county')
mortgage = dfs.get('mortgage', pd.DataFrame())   # ← was missing
loan     = dfs.get('loan',     pd.DataFrame())   # ← was missing

print(f"\n✓ Survey universe: {hh.shape[0]:,} households across {hh['a01'].nunique()} counties.")
print(f"  Individual records: {ind.shape[0]:,} people ({ind.shape[0]/hh.shape[0]:.1f} per household avg)")
print(f"  Dwelling records  : {dw.shape[0]:,} dwelling units")
print(f"  Mortgage records  : {len(mortgage):,}")
print(f"  Loan records      : {len(loan):,}")

In [ ]:
# ── 2.3  Household null audit — classify 392 columns by missingness tier ──────
null_pct = (hh.isnull().mean() * 100).sort_values(ascending=False)

tiers = {
    'Complete  (0%)     ': null_pct == 0,
    'Low       (1-20%)  ': (null_pct > 0)  & (null_pct <= 20),
    'Moderate  (21-60%) ': (null_pct > 20) & (null_pct <= 60),
    'High      (61-90%) ': (null_pct > 60) & (null_pct <= 90),
    'Extreme   (>90%)   ': null_pct > 90,
}

print("Household file — Null audit (all columns):")
print("─" * 60)
for tier, mask in tiers.items():
    count = mask.sum()
    bar   = '█' * (count // 4)
    note  = '<- structural (renter/owner split)' if 'Moderate' in tier else ''
    print(f"  {tier}  {count:>3} cols  {bar} {note}")


In [ ]:
# ── 2.4  Geographic distribution — county and residence breakdown ──────────────
hh['county_name'] = hh['a01'].map(COUNTY_MAP)
hh['residence']   = hh['a07_1'].map({1: 'Rural', 2: 'Urban'})

stratum_counts = (
    hh.groupby(['county_name', 'residence'])
      .size().unstack(fill_value=0)
      .assign(total=lambda x: x.sum(axis=1))
      .sort_values('total', ascending=False)
)

rural_total = hh['residence'].eq('Rural').sum()
urban_total = hh['residence'].eq('Urban').sum()

print("Geographic distribution summary:")
print(f"  Rural households : {rural_total:,} ({rural_total/len(hh)*100:.1f}%)")
print(f"  Urban households : {urban_total:,} ({urban_total/len(hh)*100:.1f}%)")
print(f"  Counties covered : {hh['a01'].nunique()} / 47")
print(f"  Median per county: {stratum_counts['total'].median():.0f} households")
print(f"  Range            : {stratum_counts['total'].min()} to {stratum_counts['total'].max()}")
print("\nTop 10 counties by sample size:")
print(stratum_counts[['Rural','Urban','total']].head(10).to_string())


In [ ]:
# ── 2.5  Individual file demographic profile — age, gender, education ──────────
ind['age_n']     = pd.to_numeric(ind['age_cur'], errors='coerce')
ind['gender']    = pd.to_numeric(ind['b04'],     errors='coerce').map({1:'Male', 2:'Female'})
ind['edu_isced'] = pd.to_numeric(ind.get('ken_edu_isced11',
                                          pd.Series(np.nan, index=ind.index)),
                                  errors='coerce')

age_clean = ind['age_n'].where(ind['age_n'].between(0, 100))

print("Individual file demographic profile:")
print(f"  Total individuals     : {len(ind):,}")
print(f"  Mean age              : {age_clean.mean():.1f} years")
print(f"  % Female              : {(ind['gender']=='Female').mean()*100:.1f}%")
print(f"  % Under 15 (children) : {(age_clean < 15).mean()*100:.1f}%")
print(f"  % 65+ (elderly)       : {(age_clean >= 65).mean()*100:.1f}%")
print(f"  Ages outside 0–100    : {ind['age_n'].notna().sum() - age_clean.notna().sum():,} (excluded)")
